# SPWD algorithm usability for the CQM hybrid solver (Improving CQM capacity with SPWD)

For the explanation and results of this experiment please refer to [this context](https://github.com/mkroczek/SPWD-experiments/tree/master/experiments/cqm_scheduling).

## How to run
1. clone https://github.com/wfcommons/pegasus-instances repository. Workflows from this repository will be used in the experiment.
2. set `PEGASUS_INSTANCES_DIR` to the directory containing cloned repo
3. remember to have `DWAVE_API_TOKEN` env variable to be able to use CQM
4. experiment results will be saved under the path specified with `RESULTS_FOLDER`

In [6]:
import sys
import os
from pathlib import Path
import contextlib

sys.path.append(os.path.abspath("../"))
sys.path.append(os.path.abspath("../../"))

In [7]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any

import networkx as nx
import pandas as pd
from dwave.system import LeapHybridCQMSampler
from matplotlib import pyplot as plt
import scipy.stats as stats

from QHyper.problems.algorithms.graph_utils import is_sp_dag
from QHyper.problems.algorithms.solver_utils import WorkflowSchedulingSolverDecorator, \
    DecomposedWorkflowSchedulingSolver, WorkflowSchedule
from QHyper.problems.algorithms.spization import SpIzationAlgorithm, JavaFacadeSpIzationAlgorithm
from QHyper.problems.algorithms.utils import draw
from QHyper.problems.algorithms.workflow_decomposition import SeriesParallelSplitFinal
from QHyper.problems.workflow_scheduling import Workflow, WorkflowSchedulingOneHot
from QHyper.solvers import Gurobi, solver_from_config, CQM
from QHyper.solvers.converter import Converter
from demo.reports.report import ExecutionReport, Solution

In [3]:
@dataclass
class AlgorithmRun:
    max_subgraph_size: int
    decomposition_schedule: WorkflowSchedule
    reference_schedule: WorkflowSchedule

class SolverFactory(ABC):
    def __init__(self, tasks_file, machines_file, deadline):
        self.tasks_file = tasks_file
        self.machines_file = machines_file
        self.deadline = deadline
    
    @abstractmethod
    def get_decomposed_solver(self, max_subgraph_size: int):
        pass
    
    @abstractmethod
    def get_reference_solver(self):
        pass
    
class GurobiSolverFactory(SolverFactory):
    def get_decomposed_solver(self, max_subgraph_size: int):
        workflow = Workflow(self.tasks_file, self.machines_file, self.deadline)
        division = SeriesParallelSplitFinal().decompose(workflow, max_subgraph_size)
        problems = map(lambda w: WorkflowSchedulingOneHot(w), division.workflows)
        solvers = map(lambda p: WorkflowSchedulingSolverDecorator(Gurobi(p)), problems)
        return DecomposedWorkflowSchedulingSolver(list(solvers), division)

    def get_reference_solver(self):
        solver_config = {
            "problem": {
                "type": "workflow_scheduling",
                "encoding": "one-hot",
                "tasks_file": self.tasks_file,
                "machines_file": self.machines_file,
                "deadline": self.deadline,
            },
            "solver": {
                "type": "gurobi",
            }
        }
        return WorkflowSchedulingSolverDecorator(solver_from_config(solver_config))

class CQMSolverFactory(SolverFactory):
    def __init__(self, tasks_file, machines_file, deadline, time=5):
        super().__init__(tasks_file, machines_file, deadline)
        self.time=time

    def get_decomposed_solver(self, max_subgraph_size: int):
        workflow = Workflow(self.tasks_file, self.machines_file, self.deadline)
        division = SeriesParallelSplitFinal().decompose(workflow, max_subgraph_size)
        problems = map(lambda w: WorkflowSchedulingOneHot(w), division.workflows)
        solvers = map(lambda p: WorkflowSchedulingSolverDecorator(CQM(problem=p, time=self.time)), problems)
        return DecomposedWorkflowSchedulingSolver(list(solvers), division)

    def get_reference_solver(self):
        solver_config = {
            "problem": {
                "type": "workflow_scheduling",
                "encoding": "one-hot",
                "tasks_file": self.tasks_file,
                "machines_file": self.machines_file,
                "deadline": self.deadline,
            },
            "solver": {
                "type": "cqm",
                "time": self.time
            }
        }
        return WorkflowSchedulingSolverDecorator(solver_from_config(solver_config))

class CQMDecompositionExperiment:
    class CQMExperimentResult:
        def __init__(
            self, 
            gurobi_scheduling: AlgorithmRun, 
            cqm_scheduling: AlgorithmRun,
            tasks_file: str,
            machines_file: str,
            deadline: int,
            max_subgraph_size: int
        ):
            self.gurobi_scheduling: AlgorithmRun = gurobi_scheduling
            self.cqm_scheduling: AlgorithmRun = cqm_scheduling
            self.tasks_file=tasks_file
            self.machines_file=machines_file
            self.deadline=deadline
            self.max_subgraph_size=max_subgraph_size

        def save_execution_report(self, save_dir: str):
            save_path = Path(save_dir)
            save_path.mkdir(parents=True, exist_ok=True)
        
            reports = [
                ("CQM", self.cqm_scheduling.decomposition_schedule, "cqm_decomposed"),
                ("Gurobi", self.gurobi_scheduling.decomposition_schedule, "gurobi_decomposed"),
                ("Gurobi", self.gurobi_scheduling.reference_schedule, "gurobi_raw"),
            ]
        
            for solver, schedule, suffix in reports:
                report = self._get_execution_report(solver=solver, workflow_schedule=schedule)
                report.write_json(save_path / f"{suffix}.json")

        def _get_execution_report(self, solver: str, workflow_schedule: WorkflowSchedule):
            return ExecutionReport(
                workflow_file=self.tasks_file,
                machines_file=self.machines_file,
                deadline=self.deadline,
                max_graph_size=self.max_subgraph_size,
                solver=solver,
                solution=Solution.from_workflow_schedule(workflow_schedule)
            )
        
    def __init__(self, tasks_file, machines_file, max_subgraph_size: int):
        self.tasks_file = tasks_file
        self.machines_file = machines_file
        self.deadline = deadline_as_cpv(tasks_file, machines_file)
        self.max_subgraph_size: int = max_subgraph_size
        self.gurobi_factory: SolverFactory = GurobiSolverFactory(self.tasks_file, self.machines_file, self.deadline)
        self.cqm_factory: SolverFactory = CQMSolverFactory(self.tasks_file, self.machines_file, self.deadline)

    def _run_suppress_output(self, func):
        # Helpful for running Gurobi, because it produces massive logs on stdout
        with open(os.devnull, "w") as devnull:
            with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                return func()
    
    def run(self) -> CQMExperimentResult:
        gurobi_run = AlgorithmRun(
            self.max_subgraph_size,
            self._run_suppress_output(
                lambda: self.gurobi_factory.get_decomposed_solver(self.max_subgraph_size).solve()
            ),
            self._run_suppress_output(
                lambda: self.gurobi_factory.get_reference_solver().solve()
            )
        )
        cqm_run = AlgorithmRun(
            self.max_subgraph_size, 
            self.cqm_factory.get_decomposed_solver(self.max_subgraph_size).solve(), 
            None
        )
        return self.CQMExperimentResult(
            gurobi_run, 
            cqm_run,
            self.tasks_file,
            self.machines_file,
            self.deadline,
            self.max_subgraph_size
        )

def deadline_as_cpv(tasks_file, machines_file):
    workflow = Workflow(tasks_file, machines_file, 100000)
    return int(workflow.critical_path_value)

# Actual experiment runs

In [4]:
PEGASUS_INSTANCES_DIR = "/Users/marcinkroczek/code/pegasus-instances"
RESULTS_FOLDER = "./cqm_scheduling/ec2"
MACHINES_FILE = "../workflows_data/machines/ec2_machines_normalized.json"

### Montage 310 nodes, mss = 100

In [ ]:
montage_310 = f"{PEGASUS_INSTANCES_DIR}/montage/chameleon-cloud/montage-chameleon-2mass-015d-001.json"
max_subgraph_size = 100

experiment = CQMDecompositionExperiment(montage_310, MACHINES_FILE, max_subgraph_size)
experiment_result = experiment.run()
experiment_result.save_execution_report(save_dir = f"{RESULTS_FOLDER}/montage_310_mss_100")

### Montage 472, mss = 150

In [ ]:
montage_472 = f"{PEGASUS_INSTANCES_DIR}/montage/chameleon-cloud/montage-chameleon-dss-10d-001.json"
max_subgraph_size = 150

experiment = CQMDecompositionExperiment(montage_472, MACHINES_FILE, max_subgraph_size)
experiment_result = experiment.run()
experiment_result.save_execution_report(save_dir = f"{RESULTS_FOLDER}/montage_472_mss_150")

### Montage 619, mss = 200

In [ ]:
montage_619 = f"{PEGASUS_INSTANCES_DIR}/montage/chameleon-cloud/montage-chameleon-2mass-025d-001.json"
max_subgraph_size = 200

experiment = CQMDecompositionExperiment(montage_619, MACHINES_FILE, max_subgraph_size)
experiment_result = experiment.run()
experiment_result.save_execution_report(save_dir = f"{RESULTS_FOLDER}/montage_619_mss_200")

### Montage 1066 nodes, mss = 350

In [ ]:
montage_1066 = f"{PEGASUS_INSTANCES_DIR}/montage/chameleon-cloud/montage-chameleon-dss-125d-001.json"
max_subgraph_size = 350

experiment = CQMDecompositionExperiment(montage_1066, MACHINES_FILE, max_subgraph_size)
experiment_result = experiment.run()
experiment_result.save_execution_report(save_dir = f"{RESULTS_FOLDER}/montage_1066_mss_350")

## Test only on Gurobi with small graph

In [8]:
epigenomics_small = f"{PEGASUS_INSTANCES_DIR}/epigenomics/chameleon-cloud/epigenomics-chameleon-hep-1seq-100k-001.json"
max_subgraph_size = 20

experiment = CQMDecompositionExperiment(epigenomics_small, MACHINES_FILE, max_subgraph_size)
experiment_result = experiment.run()
experiment_result.save_execution_report(save_dir = f"{RESULTS_FOLDER}/epigenomics_small")